# SEC insights pipeline runner

Thin interactive wrapper around `sec_pipeline.pipeline.run`. All logic lives in
the package so it stays testable and reusable. Configure secrets in `core/.env`
before running (see `core/README.md`).

Steps performed per airline and period:

1. Resolve the CIK from SEC (with a static fallback).
2. Download the relevant 10-Q, 10-K, and 8-K filings inside the period window.
3. Parse each filing to clean text and split it into overlapping chunks.
4. Build a local Chroma collection and retrieve the most relevant passages.
5. Generate grouped markdown insights and write them to `data/generated/insights.json`.

In [1]:
from sec_pipeline import config
from sec_pipeline.pipeline import run

print("Embedding backend:", config.EMBEDDING_BACKEND)
print("Chat model:", config.OPENAI_CHAT_MODEL)
print("Output:", config.SUMMARIES_PATH)

Embedding backend: openai
Chat model: gpt-4.1-mini
Output: C:\Users\trica\OneDrive\Documents\GitHub Repos\airline_financials\airline-dashboard\data\generated\insights.json


In [3]:
AIRLINES = ["AAL"]
YEARS = [2015]
PERIODS = ["Q1", "Q2"]

summaries = run(AIRLINES, YEARS, PERIODS, overwrite=True)

2026-09-06 22:35:24,753 INFO Processing AAL 2015Q1
2026-09-06 22:35:24,770 WARNING No filings found for AAL 2015Q1
2026-09-06 22:35:24,770 INFO Processing AAL 2015Q2
2026-09-06 22:35:24,787 WARNING No filings found for AAL 2015Q2


In [ ]:
AIRLINES = ["AAL", "DAL", "UAL", "LUV", "ALK", "JBLU", "ULCC", "ALGT", "RJET", "SKYW"]
YEARS = [2026] #list(range(2014, 2027)) #
PERIODS = ["Q3"] #["Q1", "Q2", "Q3", "Q4", "FY"]

summaries = run(AIRLINES, YEARS, PERIODS, overwrite=False)
print("Airlines summarized:", list(summaries))

2026-09-06 17:31:20,465 INFO Processing AAL 2014Q1
2026-09-06 17:31:20,481 WARNING No filings found for AAL 2014Q1
2026-09-06 17:31:20,481 INFO Processing AAL 2014Q2
2026-09-06 17:31:20,507 WARNING No filings found for AAL 2014Q2
2026-09-06 17:31:20,508 INFO Processing AAL 2014Q3
2026-09-06 17:31:20,524 WARNING No filings found for AAL 2014Q3
2026-09-06 17:31:20,525 INFO Processing AAL 2014Q4
2026-09-06 17:31:20,537 WARNING No filings found for AAL 2014Q4
2026-09-06 17:31:20,538 INFO Processing AAL 2014FY
2026-09-06 17:31:20,551 WARNING No filings found for AAL 2014FY
2026-09-06 17:31:20,551 INFO Processing AAL 2015Q1
2026-09-06 17:31:20,564 WARNING No filings found for AAL 2015Q1
2026-09-06 17:31:20,564 INFO Processing AAL 2015Q2
2026-09-06 17:31:20,576 WARNING No filings found for AAL 2015Q2
2026-09-06 17:31:20,577 INFO Processing AAL 2015Q3
2026-09-06 17:31:54,914 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-06 17:32:19,583 INFO HTTP Request

Airlines summarized: ['AAL', 'DAL', 'UAL', 'LUV', 'ALK', 'JBLU', 'ULCC', 'HA', 'SAVE', 'ALGT', 'SNCY', 'VA', 'RJET', 'SKYW']


In [4]:
# Preview one generated summary.
from IPython.display import Markdown

Markdown(summaries["AAL"]["2014"]["FY"])

### Financial Insights
1. **Record Net Income**: For the fiscal year 2014, American Airlines Group Inc. (AAG) reported a net income of $2.9 billion, a significant turnaround from a net loss of $1.8 billion in 2013. This reflects a robust recovery post-merger and improved operational efficiencies.

2. **Operating Income Growth**: AAG achieved an operating income of $4.2 billion in 2014, up from $1.4 billion in 2013. This increase of $2.8 billion was attributed to strong demand for air travel and effective cost management strategies, including a reduction in operating expenses.

3. **Special Charges**: The company recorded total net special charges of $1.3 billion in 2014, which included $824 million in operating special charges primarily related to merger integration expenses. Excluding these charges, the adjusted net income was $4.2 billion, indicating a 115% improvement compared to the previous year.

4. **Revenue Growth**: Total operating revenues for 2014 reached $42.7 billion, a 5.5% increase from $40.4 billion in 2013. This growth was driven by a 4.5% increase in mainline and regional passenger revenues, totaling $37.1 billion.

### Operational Insights
5. **Capacity Expansion**: AAG increased its total system capacity by 2.2% in 2014 compared to 2013, primarily due to the introduction of larger, more efficient aircraft. This capacity growth was essential for meeting the rising demand for air travel.

6. **Cost Management**: The mainline cost per available seat mile (CASM) excluding special items and fuel increased by 2.0% to 8.63 cents in 2014. This increase was primarily driven by higher salaries and maintenance costs associated with the integration of US Airways.

### Commercial Strategy Insights
7. **Merger Integration Success**: The merger with US Airways, completed on December 9, 2013, has shown positive results, with AAG reporting record profits in each quarter of 2014. The integration efforts included launching codesharing, aligning frequent flyer programs, and enhancing customer service across both airlines.

8. **Shareholder Returns**: AAG initiated a capital deployment program that included a $1 billion share repurchase plan and a quarterly cash dividend of $0.10 per share, marking the first dividend since 1980. This move reflects the company’s strong financial position and commitment to returning value to shareholders.

### Labor Insights
9. **Labor Relations**: AAG is currently in negotiations with various labor unions, including the Association of Professional Flight Attendants (APFA) and the International Association of Machinists (IAM). These negotiations are critical as they may impact labor costs and operational efficiencies moving forward.

### Executive Personnel Insights
10. **Leadership Stability**: Doug Parker, who serves as the Chairman and CEO of AAG, has emphasized the importance of integrating the two airlines and enhancing operational performance. His leadership has been pivotal in steering the company towards profitability and operational excellence post-merger.

### Wrap Up
In summary, American Airlines Group Inc. demonstrated a remarkable financial turnaround in 2014, achieving record profits and significant revenue growth following its merger with US Airways. The company's strategic focus on operational efficiency, capacity expansion, and shareholder returns, alongside ongoing labor negotiations, positions it well for future growth in the competitive airline industry.

In [9]:
import json
from IPython.display import Markdown
from sec_pipeline import config

with config.SUMMARIES_PATH.open(encoding="utf-8") as file:
    insights = json.load(file)

Markdown(insights["AAL"]["2014"]["FY"])

### Financial Insights

1. **American Airlines Group delivered record 2014 full-year net profit of \$4.2 billion excluding special charges, more than doubling 2013’s \$1.9 billion**  
   The company reported a net profit excluding net special charges of \$4.2 billion for 2014, up 115% from \$1.9 billion in 2013. Fourth quarter 2014 net profit excluding special charges was \$1.1 billion, a 153% increase from the prior year quarter [8-K 2015-01-27]. Full-year total operating revenues rose 59.5% to \$42.7 billion from \$26.7 billion in 2013, driven by the merger with US Airways and organic growth [8-K 2015-01-27]. Operating expenses increased as well, with fuel costs up 35.1% to \$10.6 billion and salaries, wages and benefits up 36.5% to \$8.5 billion, reflecting the combined operations and higher costs [8-K 2015-01-27].

2. **Passenger revenue per available seat mile (PRASM) increased 2.2% for 2014 but declined slightly in Q4 as capacity growth outpaced traffic**  
   Full-year consolidated PRASM was 13.97 cents, up 2.2% over 2013. However, in Q4 2014, PRASM was 13.50 cents, down 1.0% year-over-year on a 1.7% increase in ASMs, indicating some pressure on unit revenues late in the year [8-K 2015-01-27; 8-K 2015-01-12]. Consolidated passenger yield rose 4.4% for the full year and 0.9% in Q4, showing improved pricing power despite the PRASM decline [8-K 2015-01-27].

3. **Operating margins remained strong, with full-year pretax margin excluding special items around 10-13% and Q4 margin at about 10-11%**  
   The company expected and reported pretax margins excluding special charges of approximately 10-11% in Q4 2014 and 12-13% in Q2 2014 [8-K 2015-01-12; 8-K 2014-07-09]. These margins reflect profitable integration and operational scale post-merger.

4. **Robust cash position and aggressive capital return with \$1 billion share repurchase completed early and a new \$2 billion authorization**  
   At December 31, 2014, total cash and short-term investments were approximately \$8.1 billion, including \$774 million restricted cash, with an undrawn revolving credit facility of \$1.8 billion [8-K 2015-01-27]. The company repurchased 23.4 million shares in 2014 at an average price of \$42.72, completing its initial \$1 billion buyback program more than a year ahead of schedule, and authorized an additional \$2 billion buyback through 2016 [8-K 2015-01-27]. Dividends resumed with \$0.10 per share declared in Q2 and Q4 2014 [8-K 2014-07-24; 8-K 2015-01-27].

5. **Fuel costs rose sharply in 2014, increasing 35.1% year-over-year despite a modest decline in average price per gallon due to higher consumption**  
   Fuel and related taxes totaled \$10.6 billion in 2014, up from \$7.8 billion in 2013, a 35.1% increase driven by higher consumption of approximately 3.65 billion gallons [8-K 2015-01-27; 8-K 2014-04-24]. Average fuel price guidance for 2014 ranged around \$3.00 per gallon [8-K 2014-04-24], while actual prices fluctuated. The increase in fuel expense reflects both volume growth and market price volatility.

### Operations Insights

6. **System capacity increased 2.2-3.0% in 2014, led by mainline capacity growth of 2.4%, with international capacity up 4.4-7% and domestic up around 1-1.3%**  
   Total available seat miles (ASMs) for the system grew 2.2% to 2.38 trillion, with mainline ASMs up 2.4% to 237.5 billion [8-K 2015-01-12; 8-K 2014-04-08]. International capacity expansion was focused on longer stage lengths and larger gauge aircraft [8-K 2014-04-08; 8-K 2015-01-12]. Regional ASMs were roughly flat or slightly up, around 28-29 billion [8-K 2014-01-28; 8-K 2014-04-24].

7. **Revenue passenger miles (RPMs) grew modestly by 0.8% for the full year, with domestic RPMs up 1.3% but international RPMs down slightly (-1.3%)**  
   Full-year RPMs reached 195.7 billion, up 0.8% over 2013. Domestic RPMs increased 1.3% to 125.9 billion, while international RPMs declined 1.3% to 58.3 billion, reflecting weakness in Atlantic and Latin America markets partially offset by Pacific growth of 5.7% [8-K 2015-01-12]. Load factors declined 1.3 points to 82.4% for mainline, indicating capacity growth outpacing traffic [8-K 2015-01-27].

8. **Traffic trends showed seasonal and regional variation, with Pacific international traffic strong (+24.4% in December, +5.7% full year), but Atlantic and Latin America softening**  
   December 2014 RPMs on the Pacific were up 24.4% year-over-year, while Atlantic RPMs fell 7.4% and Latin America RPMs declined 6.2% [8-K 2015-01-12]. For the full year, Atlantic and Latin America were down 1.3% and 0.3% respectively, reflecting network adjustments or market conditions [8-K 2015-01-12].

9. **Load factors declined year-over-year across most regions, with overall mainline load factor down 1.3 points to 82.4% for the full year**  
   Domestic load factor was flat at 85.0%, but Atlantic and Latin America declined by 1 point or more [8-K 2015-01-27]. This was consistent with capacity growth outpacing demand, particularly internationally.

10. **Operational disruptions from severe weather in early 2014 increased cancellations by 164% compared to 2013, negatively impacting unit costs and first quarter profitability**  
    In January-February 2014, approximately 28,000 flights were canceled, a 164% increase over the same period in 2013 [8-K 2014-02-10]. The company reported this had a slightly positive impact on unit revenue but a larger negative impact on unit cost and first quarter profitability [8-K 2014-03-10].

### Labor Insights

11. **Labor costs increased significantly in 2014, with salaries, wages, and benefits rising 36.5% to \$8.5 billion, partly due to merger integration and new contracts**  
    The combined workforce grew with US Airways integration; 86% of approximately 61,600 employees at year-end were covered by collective bargaining agreements [8-K 2015-01-27; 10-K 2014-02-28]. The company reached a tentative joint collective bargaining agreement with the Association of Professional Flight Attendants [8-K 2014-11-06]. Regional pilot contracts also contributed to special charges [8-K 2015-01-27].

12. **The company made significant pension contributions in 2014, including \$600 million of supplemental contributions above the \$120 million minimum**  
    On July 23, 2014, the Board approved up to \$600 million in supplemental defined benefit plan contributions, reflecting ongoing pension funding obligations [10-Q 2014-07-24].

### Wrap Up

American Airlines Group’s 2014 fiscal year was marked by the full integration of US Airways, driving a near 60% increase in revenues and record net profits exceeding \$4 billion excluding special items. Capacity grew modestly, with a focus on international expansion, especially in the Pacific region, though international traffic was mixed with declines in Atlantic and Latin America markets. Unit revenues showed resilience with full-year PRASM up 2.2%, despite some softness in the fourth quarter. Operating margins remained solid at around 10-13%, supported by scale and operational improvements, even as fuel costs rose sharply due to increased consumption. Labor costs increased substantially from merger-related factors and new contracts, while the company maintained a strong liquidity position and aggressively returned capital to shareholders through buybacks and dividends.

The company’s 2015 guidance, as disclosed, anticipates first quarter PRASM down 2-4% and pretax margins excluding special items of approximately 12-14%, with a slight upward revision in fuel price estimates to \$1.81-\$1.86 per gallon, reflecting recent market trends [8-K 2015-02-09]. No material operational disruptions or regulatory risks beyond normal industry factors were disclosed for the period ending December 31, 2014.